# Day 23: Variational Lossy Autoencoder (VLAE)

> Xi Chen et al. (2017) - [Variational Lossy Autoencoder](https://arxiv.org/abs/1611.02731)

**Core problem:** Posterior collapse. A VAE whose decoder is a powerful autoregressive model (PixelCNN) can model the images without the latent code, and then the cheapest code in the bits-back sense ignores $z$ (Sec. 2.2).

**VLAE's answer:**
1. **Decide what $z$ must carry:** give the decoder only a small window of past pixels. Local texture can be predicted from the window; global structure (which digit, where its strokes go) cannot, so it has to come through $z$ (Sec. 3.1).
2. **Learn the prior:** an autoregressive-flow (AF) prior $p(z)$, which equals an IAF posterior along the encoder path (Sec. 3.2).
3. **Train with free bits** (App. A): the paper calls the information-preference argument "an asymptotic view" and uses 0.01 nats per pixel of free bits so the latent code is not shut off early in training.

[PATCH 2026-10-18] This notebook trains the model on binarized MNIST with free bits, measures whether $z$ is used, and repeats the run with the plain ELBO for contrast.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# [PATCH 2026-10-18] free_bits_loss / latent_usage / plot_decompressions: free bits and the z-usage checks
from implementation import VLAE, loss_function, free_bits_loss, latent_usage
from visualization import plot_reconstructions, plot_latent_sampling, plot_bits_heatmap, plot_decompressions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Technical Verification: Autoregressive Masking

VLAE relies on strict autoregressive property. We use **MADE** (Masked Autoencoder for Distribution Estimation) for the Flows and **MaskedConv2d** for the PixelCNN decoder.

In [ ]:
from implementation import MADE
dim = 10
model = MADE(dim, 32, dim)
x = torch.randn(1, dim, requires_grad=True)
out = model(x)

# Verification: Output i should NOT depend on input >= i
i = 5
out[0, i].backward()
print(f"Gradients for dimension {i}:")
print(x.grad.data.numpy())
assert torch.all(x.grad.data[0, i:] == 0)
print("Autoregressive condition MET.")

## 2. Load and Binarize MNIST

We use binarized MNIST because PixelCNN is a discrete model ($p(x_i | x_{<i})$).

In [ ]:
batch_size = 64
transform = transforms.Compose([
    transforms.ToTensor(),
    lambda x: (x > 0.5).float()
])

train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=transform),
                          batch_size=batch_size, shuffle=True)
test_loader = DataLoader(datasets.MNIST('./data', train=False, transform=transform),
                         batch_size=batch_size, shuffle=False)

# [PATCH 2026-10-18] 1,000 binarized test images as one tensor, for the z-usage checks below
test_x = (test_loader.dataset.data[:1000].float() / 255 > 0.5).float().unsqueeze(1).to(device)

## 3. Training Loop with Free Bits

[PATCH 2026-10-18] The paper trains with free bits (App. A: "0.01 nats/data-dim free bits (Kingma et al., 2016) was found to be effective in dealing with the problem of all the latent code being ignored early in training"). Free bits (Kingma et al. 2016, eq. 15) splits the latent units into groups $j$ and replaces the KL term by

$$\sum_j \max\big(\lambda,\ \mathbb{E}_{x \in \text{minibatch}}[\mathrm{KL}_j]\big),$$

so a group whose batch-average KL is below $\lambda$ costs a constant: the optimiser gains nothing by switching $z$ off, and reconstruction can use up to $\lambda$ nats per group for free. Here each of the 32 latent dimensions is a group, and the paper's 0.01 nats per pixel $\times$ 784 pixels = 7.84 nats per image becomes $\lambda$ = 7.84 / 32 = 0.245 nats per dimension. The AF prior is not factorised, so the per-dimension KL uses the chain rule: $\log q(z_d|x) - \log p(z_d \mid z_{<d})$.

The log shows the objective that is optimised and, next to it, the true KL and negative ELBO (no max), which are what the model actually achieves. Each run is 150 batches (9,600 images) on CPU; the paper trains to convergence.

In [ ]:
# [PATCH 2026-10-18] 3 full epochs = 2,814 batches of 64 at ~1.5-3 s each on one CPU core (> 1 h; hit the 480 s cap).
# Train on the first 150 batches (9,600 images). Now a function, so the plain-ELBO contrast below reuses it with
# the same initialisation and data order; free_bits=0.0 gives the plain ELBO (equal to loss_function).
max_batches = 150

def train_vlae(free_bits, max_batches=max_batches, seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)    # MADE draws its hidden-unit degrees with np.random
    model = VLAE(input_dim=1, latent_dim=32, n_layers=3, use_flow=True).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    total_obj, total_recon, total_kl, n_seen = 0, 0, 0, 0
    for batch_idx, (data, _) in enumerate(train_loader):
        if batch_idx == max_batches:
            break
        data = data.to(device)
        optimizer.zero_grad()
        logits, mu, logvar, z, log_det = model(data)
        loss, parts = free_bits_loss(model, logits, data, mu, logvar, z, free_bits=free_bits)
        loss.backward()
        optimizer.step()

        total_obj += loss.item()
        total_recon += parts['recon']
        total_kl += parts['kl']
        n_seen += len(data)
        if batch_idx % 20 == 0 or batch_idx == max_batches - 1:
            n = len(data)
            print(f"  batch {batch_idx:3d} | per image: objective {loss.item()/n:.2f} | R {parts['recon']/n:.2f} | "
                  f"KL {parts['kl']/n:.2f} | -ELBO {(parts['recon'] + parts['kl'])/n:.2f}")
    print(f"{n_seen} images | objective {total_obj/n_seen:.2f} | R {total_recon/n_seen:.2f} | "
          f"KL {total_kl/n_seen:.2f} | -ELBO {(total_recon + total_kl)/n_seen:.2f} nats per image")
    return model

print("Free bits, 0.01 nats per pixel:")
model = train_vlae(free_bits=0.01)

## 4. Is $z$ Used?

[PATCH 2026-10-18] Teacher-forced reconstructions cannot answer this (see Reconstructions below). Three measurements on 1,000 test images, in nats per image:

- **KL to the prior**, the true one (no free-bits max). It is an upper bound on the information $z$ carries.
- **The decoder's gap:** $-\log p(x|z)$ with each image's own $z \sim q(z|x)$ versus with the $z$ of a different image. If the decoder ignores $z$, both are the same.
- **$I(x; z)$:** how much one $z$ sample tells about which image it came from, estimated with the aggregate posterior over the 1,000 images (it cannot exceed $\log 1000 = 6.9$). A KL can sit at the free-bits budget without carrying information (every image mapped to the same shifted Gaussian), so the KL alone is not proof.

### Posterior Information Usage (Bits Heatmap)
The heatmap (saved to `plot4_bits_per_dim.png`) plots KL(q(z|x) || N(0, I)) per dimension, in nats. With a learned flow prior that is not the information $z$ carries: q can drift from N(0, I) toward the prior without encoding anything, so spikes do not mean the code is used. The measurements above are the honest ones.

In [ ]:
# [PATCH 2026-10-18] latent_usage: KL to the prior, -log p(x|z) with the image's own z vs another image's z, I(x; z).
# plot_bits_heatmap plots KL(q(z|x) || N(0, I)) (nats), not the KL to the learned flow prior.
def show_usage(u, label):
    print(f"{label}: KL to prior {u['kl']:.2f} | -log p(x|z): own z {u['nll_own']:.2f}, another image's z "
          f"{u['nll_shuffled']:.2f}, z ~ p(z) {u['nll_prior']:.2f} | gap {u['gap']:.2f} | "
          f"I(x;z) {u['mi']:.2f} (max {u['mi_cap']:.2f}) | -ELBO {u['neg_elbo']:.2f} nats per image")

torch.manual_seed(1)
usage_fb = latent_usage(model, test_x)
show_usage(usage_fb, "free bits ")

model.eval()
test_data, _ = next(iter(test_loader))
with torch.no_grad():
    _, mu, logvar, _, _ = model(test_x[:100])
    plot_bits_heatmap(mu, logvar)
    kl_std = (0.5 * (logvar.exp() + mu.pow(2) - 1 - logvar)).sum(1).mean().item()
print(f"KL to N(0, I) = {kl_std:.2f} nats per image (what the heatmap plots)")

### Contrast: the Plain ELBO

[PATCH 2026-10-18] The same model, initialisation and data order, trained on the plain negative ELBO (`free_bits=0.0`, identical to `loss_function`). At initialisation the AF prior is about 4x wider than $q(z|x)$, so the KL starts near 29 nats; the plain objective then pushes $q$ onto the prior, and $z$ ends up nearly unused: posterior collapse.

In [ ]:
print("Plain ELBO (no free bits):")
model_plain = train_vlae(free_bits=0.0)
torch.manual_seed(1)
usage_plain = latent_usage(model_plain, test_x)
show_usage(usage_fb, "free bits ")
show_usage(usage_plain, "plain ELBO")

### Decompressions (the Paper's Fig. 1a)

[PATCH 2026-10-18] "Decompressing" an image means $z \sim q(z|x)$, then $x' \sim p(x|z)$ sampled pixel by pixel. If $z$ carries the global structure, the decompression is the same digit with redrawn local details; if $z$ is ignored, it is just another sample. The bottom row decodes $z \sim p(z)$. After only 150 CPU batches the samples are rough; compare the two models row by row.

In [ ]:
torch.manual_seed(2)
plot_decompressions(model, test_x[:8], device, filename="plot2b_decompressions_free_bits.png",
                    title="Free bits: originals, decompressions, prior samples")
torch.manual_seed(2)
plot_decompressions(model_plain, test_x[:8], device, filename="plot2b_decompressions_plain_elbo.png",
                    title="Plain ELBO: originals, decompressions, prior samples")

### Reconstructions
These are teacher-forced one-step predictions: each pixel is predicted from the true pixels above and to its left, so they look sharp even when $z$ is ignored and do not show what $z$ encodes.

In [ ]:
plot_reconstructions(model, test_data[:8], device)

## 5. Latent Space Traversals

Does $z$ capture high-level semantics? We pick an image, encode it, and sweep one dimension of $z$ to see what changes.

In [ ]:
def plot_traversal(model, image, dim_idx, device, n_steps=8):
    model.eval()
    with torch.no_grad():
        mu, logvar = model.encoder(image.to(device))
        z = model.reparameterize(mu, logvar)
        
        traversals = []
        for val in np.linspace(-3, 3, n_steps):
            z_step = z.clone()
            z_step[0, dim_idx] += val
            
            # Fast recon (teacher forcing with zeroes for speed in visualization)
            # Correct way is autoregressive, but here we just check logits
            # [PATCH 2026-10-18] through model.decode: z now reaches the decoder as a 28x28 feature map
            logits = model.decode(torch.zeros_like(image).to(device), z_step)
            traversals.append(torch.sigmoid(logits).cpu().squeeze())
            
    fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))
    for i, img in enumerate(traversals):
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')
    plt.show()

print("Traversal for dimension 0:")
plot_traversal(model, test_data[0:1], dim_idx=0, device=device)

## 6. Full Sampling (Slow)

We generate new digits by sampling $z \sim p(z)$ (through the learned AF prior) and decoding pixel-by-pixel.

In [ ]:
plot_latent_sampling(model, device, latent_dim=32, n_samples=9)

### Summary

[PATCH 2026-10-18]
- VLAE decides what $z$ carries by restricting the decoder's window (34 past pixels here, 69 for the paper's 6 masked layers), and the decoder receives $z$ as a 28x28 feature map, as in the paper (App. A). Sec. 3.1 calls information preference "an asymptotic view"; the paper trains with free bits.
- With free bits (our run: KL 4.97 nats, gap 2.48 nats, $I(x; z)$ 0.95), $-\log p(x|z)$ is lower with an image's own $z$ than with another image's, and $I(x; z)$ is clearly above 0: $z$ is used. Free bits only removes the KL penalty below $\lambda$; it does not push the KL up to the budget.
- With the plain ELBO (our run: KL 0.71 nats, gap 0.54, $I(x; z)$ 0.16), the KL to the learned prior collapses and $z$ carries almost nothing.
- These are 150-batch CPU runs: the paper's model reaches 13.3 nats of KL and 79.03 nats NLL on statically binarized MNIST (Sec. 4.1, Table 1). This notebook thresholds pixels at 0.5 instead, which removes much of the per-pixel entropy, so its nats are not directly comparable. `train_minimal.py --max-batches 0` trains a full epoch (Gaussian prior: KL 9.5 nats, a 35-nat gap, digit probe 77 %).